In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [5]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
batch_size = 100
# batch_size = None

# opt = torch_numopt.GaussNewton(model=model, lr=1, line_search_method="const")
opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", batch_size=batch_size)
# opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method='backtrack', line_search_cond='wolfe')
# opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method='backtrack', line_search_cond='strong-wolfe')
# opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method='backtrack', line_search_cond='goldstein')

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5360604524612427
epoch:  1, loss: 0.3733004629611969
epoch:  2, loss: 0.2780858278274536
epoch:  3, loss: 0.16858315467834473
epoch:  4, loss: 0.11306289583444595
epoch:  5, loss: 0.10828884690999985
epoch:  6, loss: 0.06401661038398743
epoch:  7, loss: 0.06063719466328621
epoch:  8, loss: 0.057345304638147354
epoch:  9, loss: 0.04356943815946579
epoch:  10, loss: 0.0254893247038126
epoch:  11, loss: 0.017894523218274117
epoch:  12, loss: 0.016254005953669548
epoch:  13, loss: 0.015039889141917229
epoch:  14, loss: 0.014092903584241867
epoch:  15, loss: 0.01335487887263298
epoch:  16, loss: 0.012820265255868435
epoch:  17, loss: 0.011693010106682777
epoch:  18, loss: 0.01080961711704731
epoch:  19, loss: 0.010155283845961094
epoch:  20, loss: 0.009630742482841015
epoch:  21, loss: 0.009188481606543064
epoch:  22, loss: 0.008775238879024982
epoch:  23, loss: 0.008429132401943207
epoch:  24, loss: 0.008086975663900375
epoch:  25, loss: 0.007763152942061424
epoch:  26, 

In [7]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.915955651802528
Test metrics:  R2 = 0.49198902803844335
